# KLE Long Training Colab Notebook

这个 notebook 专门用于“训练更久 + 回测更长窗口”。

## 目标
- 跑更长窗口（`last=120`）
- 跑更高随机对照样本（`random_trials=500`）
- 跑覆盖策略比较（`N=10,20,30,40,50`）
- 跑更重的 FastRF 参数（更多树、更深）

> 说明：该 notebook 运行时间会明显更长（可能 20~60 分钟）。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q -U pip")
run("python -m pip install -q -r /content/lotto-image-predictor-research/requirements.txt || true")

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")

# 再次安装，确保 clone 后 requirements 生效
run(f"python -m pip install -q -r {REPO_ROOT / 'requirements.txt'}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {Path.cwd()}")

In [ ]:
import pandas as pd

data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.tail(3))

## Step 1: 长窗口策略排行榜（含 FastRF）

这一步会运行 `benchmark`，对多个策略在同一窗口下做比较。

In [ ]:
# 长训练参数（你可以按需调整）
LAST = 120
RANDOM_TRIALS = 500
TREES = 180
DEPTH = 11
LEAF = 3

cmd_benchmark = (
    f'python "scripts/ml_predict_v2.py" benchmark '
    f'--last {LAST} --random-trials {RANDOM_TRIALS} '
    f'--trees {TREES} --depth {DEPTH} --leaf {LEAF}'
)
print(cmd_benchmark)
run(cmd_benchmark)

## Step 2: 覆盖策略长窗口回测（best-of-N）

重点看：
- `P>=10`
- `P>=8`
- uplift CI 是否覆盖 0

In [ ]:
COVER_SETS = "10,20,30,40,50"
DETAIL_N = 40

cmd_cover = (
    f'python "scripts/ml_predict_v2.py" realistic-cover-backtest '
    f'--last {LAST} --random-trials {RANDOM_TRIALS} '
    f'--cover-sets {COVER_SETS} --seed 42 '
    f'--print-picks --detail-n {DETAIL_N}'
)
print(cmd_cover)
run(cmd_cover)

## Step 3: 可选超长压力测试

如果你愿意进一步验证稳定性，可以把窗口拉到 180 或 240（非常慢）。

In [ ]:
# 可选：取消注释执行
# LAST_ULTRA = 180
# cmd_ultra = (
#     f'python "scripts/ml_predict_v2.py" realistic-cover-backtest '
#     f'--last {LAST_ULTRA} --random-trials 300 --cover-sets 20,30,40 --seed 42'
# )
# print(cmd_ultra)
# run(cmd_ultra)